# Baseline ML Models

The objective of this notebook is to establish solid performance baselines using a stratified train/test split. All preprocessing logic is confined strictly within the training pipeline to prevent any potential data leakage.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('..'))
from src.features.build_features import (
    FeatureEngineer, 
    RAW_NUMERICAL_FEATURES, 
    RAW_CATEGORICAL_FEATURES,
    ENGINEERED_NUMERICAL_FEATURES,
    ENGINEERED_CATEGORICAL_FEATURES
)

df = pd.read_csv('../data/processed/cleaned_telco_churn.csv')

## 1. Train/Test Split

We perform a stratified split (80/20) to ensure the target churn distribution remains identical in both sets.

In [ ]:
X = df.drop(columns=['churn_value', 'customerid'], errors='ignore')
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 2. Naive Baseline (Majority Class)

A naive model simply predicts that *no one* will churn (since retention is the majority class at ~73.5%).

In [ ]:
y_pred_naive = np.zeros_like(y_test)

naive_acc = accuracy_score(y_test, y_pred_naive)
naive_prec = precision_score(y_test, y_pred_naive, zero_division=0)
naive_rec = recall_score(y_test, y_pred_naive, zero_division=0)
naive_f1 = f1_score(y_test, y_pred_naive, zero_division=0)
naive_roc_auc = roc_auc_score(y_test, y_pred_naive)
naive_pr_auc = average_precision_score(y_test, y_pred_naive)

print(f"Naive Accuracy: {naive_acc:.3f}")
print(f"Naive Precision: {naive_prec:.3f}")
print(f"Naive Recall: {naive_rec:.3f}")
print(f"Naive F1: {naive_f1:.3f}")
print(f"Naive ROC-AUC: {naive_roc_auc:.3f}")
print(f"Naive PR-AUC: {naive_pr_auc:.3f}")

## 3. Logistic Regression Baseline

Logistic Regression is the gold standard baseline for binary classification. It is highly interpretable, fast to train, resistant to overfitting, and provides well-calibrated probabilities. 

We wrap our custom `FeatureEngineer` and standard scaling/encoding inside a scikit-learn `Pipeline` to ensure all transformations are fitted *only* on `X_train`.

In [ ]:
all_numerical = RAW_NUMERICAL_FEATURES + ENGINEERED_NUMERICAL_FEATURES
all_categorical = RAW_CATEGORICAL_FEATURES + ENGINEERED_CATEGORICAL_FEATURES

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), all_numerical),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), all_categorical)
    ],
    remainder='drop'
)

lr_pipeline = Pipeline([
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Fit on training data ONLY
lr_pipeline.fit(X_train, y_train)

# Predict on test data
y_pred_lr = lr_pipeline.predict(X_test)
y_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_prec = precision_score(y_test, y_pred_lr)
lr_rec = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_roc_auc = roc_auc_score(y_test, y_proba_lr)
lr_pr_auc = average_precision_score(y_test, y_proba_lr)

print(f"LR Accuracy: {lr_acc:.3f}")
print(f"LR Precision: {lr_prec:.3f}")
print(f"LR Recall: {lr_rec:.3f}")
print(f"LR F1: {lr_f1:.3f}")
print(f"LR ROC-AUC: {lr_roc_auc:.3f}")
print(f"LR PR-AUC: {lr_pr_auc:.3f}")